# 📊 Chapter 6: Random Variables
*Tier 1 — Foundations | All Tracks*

> **🎬 Hook:** A "random variable" isn't really random at all — it's the most precise tool in probability.
> It's a function that maps every outcome to a number.

**🎯 Objectives:** Understand discrete vs continuous RVs, PMF, PDF, and CDF.

## 📖 Section 1 — Concept Review

A **Random Variable X** is a function: X : Ω → ℝ (maps outcomes to numbers).

### Discrete vs Continuous
| | **Discrete** | **Continuous** |
|--|--|--|
| Values | Countable (0,1,2,3,...) | Uncountable interval |
| Description | **PMF** P(X=x) | **PDF** f(x) |
| Cumulative | CDF F(x) = P(X ≤ x) | CDF F(x) = ∫f(t)dt |
| Example | Number of goals | Height, weight, time |

### PMF (Probability Mass Function)
$$P(X = x) \geq 0 \quad \text{and} \quad \sum_x P(X=x) = 1$$

### PDF (Probability Density Function)
$$f(x) \geq 0 \quad \text{and} \quad \int_{-\infty}^{\infty} f(x)dx = 1$$

⚠️ For continuous RVs, P(X = exact value) = 0. We always compute P(a ≤ X ≤ b).

## 🧮 Section 2 — Math Walkthrough

### Discrete Example: Die Roll
X = face value of a fair die. PMF: P(X=k) = 1/6 for k ∈ {1,2,3,4,5,6}

### CDF of die roll:
$$F(x) = P(X \leq x) = \frac{\lfloor x \rfloor}{6} \quad \text{for } 1 \leq x \leq 6$$

### Continuous Example: Uniform on [0,1]
$$f(x) = 1 \quad \text{for } 0 \leq x \leq 1$$
$$P(0.3 \leq X \leq 0.7) = \int_{0.3}^{0.7} 1 \, dx = 0.4$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
np.random.seed(42)

fig = plt.figure(figsize=(14, 8))
gs = gridspec.GridSpec(2, 3, figure=fig)

# ── Discrete: Die Roll PMF and CDF ──
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[1, 0])
die_vals = np.arange(1, 7)
pmf = np.ones(6) / 6
ax1.bar(die_vals, pmf, color='#3498db', edgecolor='white', lw=2)
ax1.set_title('🎲 Die Roll — PMF', fontweight='bold')
ax1.set_xlabel('X'); ax1.set_ylabel('P(X=x)')
ax1.set_ylim(0, 0.25)

cdf = np.cumsum(pmf)
ax2.step(die_vals, cdf, where='post', color='#3498db', lw=2.5)
ax2.scatter(die_vals, cdf, color='#3498db', s=80, zorder=5)
ax2.set_title('Die Roll — CDF', fontweight='bold')
ax2.set_xlabel('X'); ax2.set_ylabel('F(x) = P(X≤x)')

# ── Continuous: Normal PDF and CDF ──
ax3 = fig.add_subplot(gs[0, 1])
ax4 = fig.add_subplot(gs[1, 1])
x = np.linspace(-4, 4, 300)
pdf = stats.norm.pdf(x)
cdf_norm = stats.norm.cdf(x)

ax3.plot(x, pdf, color='#e74c3c', lw=3)
ax3.fill_between(x, pdf, where=(x >= -1) & (x <= 1), alpha=0.3, color='#e74c3c',
                 label='P(-1 ≤ X ≤ 1) = 68.3%')
ax3.set_title('📊 Normal — PDF', fontweight='bold')
ax3.set_xlabel('X'); ax3.set_ylabel('f(x)')
ax3.legend(fontsize=8)

ax4.plot(x, cdf_norm, color='#e74c3c', lw=3)
ax4.axhline(0.5, color='gray', linestyle='--', lw=1)
ax4.set_title('Normal — CDF', fontweight='bold')
ax4.set_xlabel('X'); ax4.set_ylabel('F(x) = P(X≤x)')

# ── Poisson: discrete, for counts ──
ax5 = fig.add_subplot(gs[0, 2])
ax6 = fig.add_subplot(gs[1, 2])
poisson = stats.poisson(mu=3)
k = np.arange(0, 13)
ax5.bar(k, poisson.pmf(k), color='#27ae60', edgecolor='white', lw=1.5)
ax5.set_title('🔢 Poisson(λ=3) — PMF', fontweight='bold')
ax5.set_xlabel('k'); ax5.set_ylabel('P(X=k)')

ax6.step(k, poisson.cdf(k), where='post', color='#27ae60', lw=2.5)
ax6.set_title('Poisson — CDF', fontweight='bold')
ax6.set_xlabel('k'); ax6.set_ylabel('F(k)')

plt.suptitle("PMF vs PDF vs CDF: Discrete and Continuous Random Variables", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('ch06_rv_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔬 Section 3 — Simulation: Build PMF and PDF from data

In [ ]:
# Simulate and recover the distribution
np.random.seed(42)

# Discrete: count of emails per hour (Poisson)
lambda_true = 5
email_counts = np.random.poisson(lambda_true, size=1000)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Empirical PMF vs Theoretical
k = np.arange(0, 16)
empirical_pmf = [(email_counts == ki).mean() for ki in k]
theoretical_pmf = stats.poisson.pmf(k, lambda_true)

axes[0].bar(k - 0.2, empirical_pmf, 0.4, label='Empirical', color='#3498db', alpha=0.7)
axes[0].bar(k + 0.2, theoretical_pmf, 0.4, label='Theoretical', color='#e74c3c', alpha=0.7)
axes[0].set_title(f'📧 Emails/Hour: Empirical vs Poisson(λ={lambda_true})', fontweight='bold')
axes[0].set_xlabel('Count'); axes[0].set_ylabel('Probability')
axes[0].legend()

# Continuous: height data (Normal)
heights = np.random.normal(170, 10, 500)
axes[1].hist(heights, bins=30, density=True, alpha=0.6, color='#3498db', label='Data (n=500)')
x = np.linspace(135, 205, 200)
axes[1].plot(x, stats.norm.pdf(x, 170, 10), color='#e74c3c', lw=3, label='Normal(170,10)')
axes[1].set_title('📏 Heights: Histogram vs PDF', fontweight='bold')
axes[1].set_xlabel('Height (cm)'); axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.savefig('ch06_simulation.png', dpi=150, bbox_inches='tight')
plt.show()

print("Key observations:")
print(f"  Max empirical PMF at k={np.argmax(empirical_pmf)}, theoretical at k={np.argmax(theoretical_pmf)}")
print(f"  Heights: mean={heights.mean():.1f}, std={heights.std():.1f} (true: 170, 10)")

## 📂 Section 5 — Real Exercise: CDF for Decision Making

Use the CDF to answer: "What fraction of students score below 75?"

In [ ]:
# Exam scores: use CDF to answer probability questions
np.random.seed(42)
exam_scores = np.random.normal(68, 12, 500)
exam_scores = np.clip(exam_scores, 0, 100)

# Fit a normal distribution
mu, sigma = exam_scores.mean(), exam_scores.std()
dist = stats.norm(mu, sigma)

questions = [
    ("P(score < 75)",   dist.cdf(75)),
    ("P(score > 80)",   1 - dist.cdf(80)),
    ("P(60 < score < 80)", dist.cdf(80) - dist.cdf(60)),
    ("Score at 90th percentile", dist.ppf(0.90)),
]

print("📝 Exam Score Analysis (Normal distribution fit)")
print(f"  Mean = {mu:.1f}, Std = {sigma:.1f}")
print()
for q, ans in questions:
    print(f"  {q} = {ans:.3f}")

x = np.linspace(30, 105, 300)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(exam_scores, bins=30, density=True, alpha=0.5, color='#3498db', label='Scores')
ax1.plot(x, dist.pdf(x), 'r-', lw=3, label=f'N({mu:.0f},{sigma:.0f})')
ax1.axvline(75, color='green', linestyle='--', lw=2, label='Score=75')
ax1.fill_between(x, dist.pdf(x), where=(x < 75), alpha=0.2, color='green')
ax1.set_title('PDF: Shaded area = P(score < 75)', fontweight='bold')
ax1.legend(fontsize=8)

ax2.plot(x, dist.cdf(x), 'r-', lw=3)
ax2.axvline(75, color='green', linestyle='--', lw=2)
ax2.axhline(dist.cdf(75), color='green', linestyle=':', lw=2)
ax2.scatter([75], [dist.cdf(75)], s=100, color='green', zorder=5)
ax2.set_title('CDF: F(75) = P(score ≤ 75)', fontweight='bold')
ax2.set_xlabel('Score'); ax2.set_ylabel('F(x)')
plt.tight_layout()
plt.savefig('ch06_exam_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## ✏️ Section 6 — Practice Problems

**P1:** X ~ Poisson(λ=4). Find P(X=2), P(X≤3), P(X>5).
**P2:** X ~ N(100, 15). Find P(X > 130), P(85 < X < 115), the 95th percentile.
**P3:** Why is P(X = 1.5) = 0 for a continuous random variable?

<details><summary>💡 Solutions</summary>

**P1:** P(X=2) = e⁻⁴·4²/2! ≈ 0.147, P(X≤3) ≈ 0.433, P(X>5) ≈ 0.215

**P2:** P(X>130) = P(Z>2) ≈ 0.023, P(85<X<115) ≈ 0.683, 95th pct = 100+1.645×15 ≈ 124.7

**P3:** Continuous distributions have infinitely many possible values — any single point has zero "width" and thus zero probability. Only intervals have positive probability.
</details>

In [ ]:
# Solutions
print("P1: Poisson(λ=4)")
print(f"  P(X=2)  = {stats.poisson.pmf(2, 4):.4f}")
print(f"  P(X≤3)  = {stats.poisson.cdf(3, 4):.4f}")
print(f"  P(X>5)  = {1 - stats.poisson.cdf(5, 4):.4f}")
print()
print("P2: Normal(100, 15)")
print(f"  P(X>130)       = {1 - stats.norm.cdf(130, 100, 15):.4f}")
print(f"  P(85<X<115)    = {stats.norm.cdf(115,100,15) - stats.norm.cdf(85,100,15):.4f}")
print(f"  95th percentile = {stats.norm.ppf(0.95, 100, 15):.2f}")

## 🎯 Recap
1. RVs map outcomes to numbers — discrete (PMF) or continuous (PDF).
2. CDF tells you P(X ≤ x) — use it to answer real questions.
3. For continuous RVs, P(X = exact value) = 0; only intervals have probability.

**Next:** [Chapter 7 — Expected Value & Variance]